# Build email extractor 

In [1]:
from pydantic import BaseModel, Field, EmailStr
from typing import Optional, Literal
from pydantic_ai import Agent
from constants import MODEL_LARGE
from dotenv import load_dotenv

load_dotenv()

class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["damadged product", "billing", "technical", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="summarize the email in 3-4 sentences.")

email_extractor_agent = Agent(MODEL_LARGE, system_prompt="""
You are customer support agent, your task is to 
extract relevant information from an email.
""", output_type= EmailExtractor)


TODO: Load in the data and test the agent on one email

In [6]:
import pandas as pd

df = pd.read_json("data/emails_cleaned.json")
df

,inputs,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea..."


In [7]:
sample_mail = df.iloc[2]["inputs"]["email"]
sample_mail

"From: Oscar Johansson <oscar.johansson@yahoo.se>\nSubject: Cannot access my account for 3 days - Urgent help needed\n\nHello Support Team,\n\nI am reaching out because I have been completely locked out of my account for the past three days and I am running out of ideas on how to fix this on my own. The problem started on Monday evening when I tried to log in as usual but kept receiving an 'Invalid credentials' error despite being absolutely certain that I was entering the correct password.\n\nI followed the instructions on your website to reset my password, but the problem is that the password reset email never arrives in my inbox. I have checked my spam and junk folders multiple times, and there is nothing there either. I have attempted the reset process at least six or seven times across different browsers and even from my phone, but the result is always the same - no email arrives.\n\nThis is causing me real problems because I have important documents and data stored in my account 

In [15]:
result = await email_extractor_agent.run(sample_mail)

result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary='Oscar Johanson cannot access his account for three days, receiving invalid credentials errors. Password reset emails never arrive despite multiple attempts and checks of spam folders. He needs urgent access to important work documents before a Friday deadline and is willing to verify his identity.')

TODO: show ground truth and compare

## Load in prompts from mlflow

In [ ]:
from mlflow.genai import load_prompt


class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["damadged product", "billing", "technical", "other"]
    urgency: Literal["low", "medium", "high"] = Field(
        description=load_prompt("email-urgency-description").format()
    )
    summary: str = Field(
        description=load_prompt("summary-description").format(num_sentences=4)
    )


email_extractor_agent = Agent(
    MODEL_LARGE,
    system_prompt=load_prompt("email-extractor-system-prompt").format(),
    output_type=EmailExtractor,
)

In [ ]:
result = await email_extractor_agent.run(sample_mail)
result.output